In [7]:
### Importation

import subprocess
subprocess.run(["pip", "install", "scikit-dimension", "-q"])

import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

###Définition

def levina_bickel(data, k=10):
    
    nn = NearestNeighbors(n_neighbors=k+1)  #détermination des nearest neighbors 
    nn.fit(data)
    distances, _ = nn.kneighbors(data)
    distances = distances[:, 1:]
    #print(distances)
    r_k = distances[:, -1]  #take the neighbor the most far away
    
    estimates = []
    
    for i in range(len(data)):
        if r_k[i] > 0:
            ratio_logs = np.log(r_k[i] / distances[i, :-1])
            if np.all(ratio_logs > 0):
                estimates.append((k - 1) / np.sum(ratio_logs))
    
    return np.mean(estimates)



def run_and_time(method, data, **kwargs):
    start   = time.time()
    result  = method(data, **kwargs)
    elapsed = time.time() - start
    return result, elapsed

N = 2000

proof_cases    = {}
proof_expected = {}

# Case 1: straight line (FD should be 1)
t = np.random.uniform(0, 10, N)
proof_cases['Straight line']      = np.column_stack([t, 2*t + 1])
proof_expected['Straight line']   = 1.0

# Case 2: line with noise (FD should be slightly > 1)
t = np.random.uniform(0, 10, N)
proof_cases['Noisy line']         = np.column_stack([t, 2*t + 1 + 0.5*np.random.normal(0,1,N)])
proof_expected['Noisy line']      = 1.2  # approximate

# Case 3: smooth surface (FD should be 2)
x = np.random.uniform(0, 5, N)
y = np.random.uniform(0, 5, N)
proof_cases['Smooth surface']     = np.column_stack([x, y, np.sin(x*y)])
proof_expected['Smooth surface']  = 2.0

# Case 4: noisy surface (FD should be slightly > 2)
x = np.random.uniform(0, 5, N)
y = np.random.uniform(0, 5, N)
proof_cases['Noisy surface']      = np.column_stack([x, y, np.sin(x*y) + 0.3*np.random.normal(0,1,N)])
proof_expected['Noisy surface']   = 2.3  # approximate

# Case 5: completely random (FD should be 3)
proof_cases['Random (x,y,z)']     = np.random.uniform(0, 5, (N, 3))
proof_expected['Random (x,y,z)']  = 3.0

print("Computing FD with Two-NN on all proof cases...\n")
print(f"{'Case':<20} {'Expected':>10} {'Measured':>10} {'Difference':>12} {'Conclusion'}")
print("-" * 72)

###Tests

for name, data in proof_cases.items():
    fd, _  = run_and_time(levina_bickel, data)
    diff   = fd - proof_expected[name]
    print(f"{name:<20} {proof_expected[name]:>10.1f} {fd:>10.3f} {diff:>+12.3f}")


Computing FD with Two-NN on all proof cases...

Case                   Expected   Measured   Difference Conclusion
------------------------------------------------------------------------
Straight line               1.0      1.123       +0.123
Noisy line                  1.2      2.325       +1.125
Smooth surface              2.0      2.321       +0.321
Noisy surface               2.3      2.958       +0.658
Random (x,y,z)              3.0      3.234       +0.234
